# Linguistic Variation in Subreddit Communities

Authors: Mounami Kayitha, Varun Sardana, Prithvi Rao, Gabor Szita, Srishti Thapar

## Introduction

Late 2022 saw the release of ChatGPT, an LLM that become one of the first to introduce AI to a mainstream audience. Since then, public discourse around AI has exploded, especially in online communities. One such large forum is Reddit. Various subreddits developed to allow users to interact and debate on the topic of AI. However, these subreddits also developed distinct cultures around the same topic. To explore this variation, we explored the relationships with AI for four subreddits: r/MachineLearning, r/artificial, r/singularity, and r/Futurology. Despite sharing a common topic, these communities appear to attract meaningfully different audiences, leading to questions of whether the language and focus of each subreddit is unique. To that end, we sought to answer whether a machine learning model can accurately classify which subreddit a post belongs to based solely on its text, and what does classification performance tell us about how these communities differ linguistically? We compared two NLP representations: TF-IDF bag of words via three classifiers, and fine-tuned BERT transformer models. Accounting for class imbalance, both approaches exceeded the baseline, with BERT reaching 75–76% accuracy and macro F1 of 0.72–0.73. From these relatively high results, we find that despite all four subreddits discussing AI, they attract different posters using different vocabularies, concerns, and ways of framing the general topic at hand.



## Methods

### Data Collection and Cleaning

Data was collected from four, finalized subreddits with high relevance to AI using Reddit's public JSON API. After initial surveying and hitting ~250 post pagination cap per endpoint, multiple endpoints were queried to maximize dataset size: current year, all time top posts, currently trending (hot), and most controversial posts of the past year. Duplicates were removed via unique post ID and only self-text posts (text only, no links, no images) that had body text were kept. Our time period was capped from January 2023 to May 2026 (collection day), to ensure posts are after ChatGPT's releases.


### Data Preprocessing
Title and body were combined before removing URLS, non-alphabetic characters, extra whitespace which further cleaned the data. Standard English stopwords were removed as well as an extended list of Reddit-specific words that only provided a source of noise. Each token was lemmatized such that they are reduced to their base using WordNetLemmatizer.


The final dataset contained 2972 posts across four subreddits:

- r/MachineLearning: 984 posts
- r/artificial: 782 posts
- r/Futurology: 583 posts
- r/singularity: 578 posts

Each of these observations represents a single Reddit post and includes information such as the subreddit, title, body text, score, upvote ratio, number of comments, date created, and combined text.

As shown in Figure 1, the distribution of posts in each subreddit class was not balanced. r/Machine Learning had roughly 1.7x more posts than r/singularity. As a result, a naive classifer that always predicted a post to be from Machine Learning would achieve 33.6% accuracy without any real training. Therefore, a macro F1 score was used as the primary metric for evaluating model strength by averaging F1 scores across all classes.

![posts-per-subreddit.png](posts-per-subreddit.png)

*Posts Per Subreddit. r/Machine Learning (987) had roughly 1.7x more posts than r/singularity (578).*

### Exploratory Data Analysis

These four subreddits both differ and are similar in various other metrics. Figure 2 shows the similarities in post lengths across the communities, mostly around the 250 mark with some outliers surpassing 2500.

![posts-length.png](posts-length.png)
*Figure 2: Length of posts does not vary drastically by community.*

Interestingly, there is a high influx of posting activity across subreddits between 2:00PM UTC and 4:00PM UTC, with r/MachineLearning having a much higher post volume (Figure 3).

*Figure 3:*
![post-activity-hour.png](post-activity-hour.png)

### TF-IDF Feature Extraction

After cleaning, we then converted posts into numeric features using TF_IDF vectorization. This provided higher weightage to words and phrases that are important in a post but not common across the entire dataset. The vectorizer was set up with 15,000 max features (both unigram and bigram) and a minimum and maximum document frequency to remove rare and common terms. This procuded a 2,927 × 15,000 sparse matrix that is 99.23% sparse.

The data was split into an 80/20 training-test with stratification by subreddit. This resulted in 2,341 posts for training and 586 posts for testing. Stratification was used due to the subreddit class uneven distribution.


Before training, we explored the vocabulary commonalities between subreddits (Figure 4).

*Figure 4: The following TF-IDF matrix shows the feature overlap between classes.*
![tfidfmatrix.png](tfidfmatrix.png)

We took the top 100 features per class and counted how many features each pair of subreddits share. The result is shown as a lower-triangle heatmap.

The diagonal is always 100 (a class shares all 100 of its own features with itself). The interesting numbers are the off-diagonal ones. A high number like 60 means two communities share 60% of their most characteristic features, meaning they are linguistically similar and the classifier will struggle to tell them apart. A low number like 20 means only 20% overlap (well-separated, easy to classify).

We can see in the confusion matrix that singularity and artificial share the most features, and Futurology and MachineLearning share the least features (most different audiences). Therefore, we expect that singularity and artificial to be the hardest for the models to tell apart, and Futurology and MachineLearning to be the easiest for the models to tell apart.

### Models
Three classifiers were trained using the TF-IDF matrix, Multinomial Naive Bayes, Logistic Regression, and LinearSVC. These models train on word frequencies where Naive bayes assumes feature independence, LogReg applies weight training (parameters: c=2, class_weight = “balanced”), and LinearSVC classifies via high-dimensional boundaries. These models were used as both baselines (Naive Bayes) and because some like LinearSVC are known for performing well in high-dimensional, sparse datasets.

To test the importance of contextual information, BERT based classifiers were fine-tuned (BERT and DistilBERT). BERT reads the full token "sentence" to produce representations of meaning with respect to context. This allows the model to integrate contextual cues for classification.

We also fine-tuned two BERT models to classify the reddit posts: `bert-base-uncased` and `distilbert-base-uncased`. These models are far more powerful than the simple TF-IDF-based models above.

Training both models allowed us to determine key characteristics of the subreddits including subtopical variation and linguistic differences.

### Results

### 
r/MachineLearning featured more academic terms like model, training, paper, and llm. r/Futurology leans toward more generic terms like human, future and world. r/artificial concentrated on LLM products like claude and chatgpt, and r/singularity uses vocab like agi, gpt. TF-IDF analysis confirmed that each subreddit has a somewhat unique vocabulary. Figure 7 describes the top 15 features per subreddit across training posts of each class and showcases the unique vocabulary we expected to see and would indicate high model classification macro F1. There is some overlap in vocabulary that is ranked high, like "model", which is a byproduct of the TF-IDF method used with a low document count.

*Figure 5:*
![posts-vocabulary.png](posts-vocabulary.png)


All three TF-IDF classifiers were evaluated to exceed both the random baseline (25%) and majority-class baseline (33.6%), confirming that each subreddit's postings have sufficient for distinguishing them. Looking at Figure 9a, r/singularity and r/artifical are commonly mixed with one another which is a trend across all models. r/Futurology improves with complexity and r/MachineLearning has consistently high F1 scores.


*Figure 6a: Results for the three TF-IDF models:*

![tfidf-confusion-matrices.png](tfidf-confusion-matrices.png)

*Figure 6b: Results for the BERT models:*

![bert-confusion-matrices.png](bert-confusion-matrices.png)

## Conclusion 

We trained five models to classify Reddit posts across four AI subreddits: Naive Bayes (macro F1 = 0.594), Logistic Regression (F1 = 0.650), LinearSVC (67.2% accuracy, F1 = 0.640), and two fine-tuned BERT models (75-76% accuracy, F1 = 0.72-0.73).  All models produced well above the 33.6% majority-class baseline. `distilbert-base-uncased` matched the accuracy of`bert-base-uncased` and even performed slightly better, despite being 40% smaller, making it the best practical choice for efficiency and accuracy.

Per-class results were consistent across all models: r/MachineLearning was easiest due to its distinctive vocabulary, and r/singularity was hardest because it overlaps linguistically with every other community. The r/artificial-r/singularity confusion was the most common error across all approaches, a pattern we predicted during the EDA before any of the models were trained as seen with the 50% vocabulary overlap.

A key limitation of this analysis is that we did not control for topic across subreddits. This could mean the classifier may be learning topical differences in some cases rather than purely linguistic ones, and we cannot separate the two without extremely limiting the dataset. Further work could be to explore feature interpretability analysis. Identifying which words and phrases most drive classification per subreddit, via dimension reduction, could also us to separate whether semantics or topics are what separate these subreddits.

Overall, classification and exploration of these subreddits supports the idea that r/MachineLearning, r/singularity, r/artifical, r/Futurology, albeit related to the topic of AI, do exhibit linguistic differences that can be detected by both complex and classical NLP models.
